# OncoResolve: N-of-1 Patient Uniqueness Scoring (CUS)

In precision oncology, subtyping patients into general categories is not always sufficient. A patient may have a highly atypical or unique genomic expression profile that requires custom targeted therapy.

This notebook demonstrates how to calculate the **Composite Uniqueness Score (CUS)** to identify cohort outliers. CUS combines two orthogonal metrics:
1. **Topological Distance:** Euclidean distance of the patient's profile from the cohort average.
2. **Reconstruction Error ($1 - R^2$):** How poorly a Ridge regression model reconstructs the patient's expression profile using the profiles of the other patients in the cohort.

In [ ]:
import numpy as np
import pandas as pd
import OncoResolve as orr
import matplotlib.pyplot as plt

print(f"OncoResolve version: {orr.__version__}")

## Step 1: Generate Simulated Cohort Data

We will generate expression profiles for 100 patient samples across 50 genes. We will purposely inject **one extreme outlier patient** and **two moderate outlier patients** to see if the CUS algorithm can detect them.

In [ ]:
np.random.seed(123)
n_patients = 100
n_genes = 50

# Create typical cohort data (mean=0, std=1)
cohort_data = np.random.normal(loc=0.0, scale=1.0, size=(n_patients, n_genes))

# Inject anomalies:
# Patient 0 is an extreme outlier (highly elevated genes across the board)
cohort_data[0, :] += np.random.normal(loc=4.5, scale=1.5, size=n_genes)

# Patient 10 is a moderate outlier (reversed expression profiles)
cohort_data[10, :] *= -3.0

barcodes = [f"PATIENT-{i:03d}" for i in range(n_patients)]
df_expr = pd.DataFrame(cohort_data, index=barcodes, columns=[f"GENE-{i:02d}" for i in range(n_genes)])

# Pre-scale cohort (required for CUS)
df_expr_scaled = orr.scale_cohort(df_expr)
print(f"Simulated Cohort Dimensions: {df_expr_scaled.shape}")

## Step 2: Compute Composite Uniqueness Scores (CUS)

We will run `compute_cus` to obtain the topological distance, reconstruction error, and unified CUS index (scaled from 0 to 1).

In [ ]:
# Compute CUS scores
df_cus = orr.compute_cus(df_expr_scaled, barcodes=df_expr_scaled.index, alpha=0.01)

print("Top 10 Most Unique (Outlier) Patients:")
df_cus.head(10)

## Step 3: Analyze the Results

As we can see, the injected outliers **`PATIENT-000`** and **`PATIENT-010`** were successfully flagged with the highest CUS scores! Let's plot the results.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df_cus["Topo_Distance"], df_cus["Recon_Error"], c=df_cus["CUS"], cmap="coolwarm", s=100, edgecolors='k')

# Annotate top outliers
for idx, row in df_cus.head(3).iterrows():
    plt.annotate(row["Patient_ID"], (row["Topo_Distance"], row["Recon_Error"]), 
                 textcoords="offset points", xytext=(5, 5), ha='left', weight='bold')

plt.title("N-of-1 Patient Uniqueness Space", fontsize=14, weight='bold')
plt.xlabel("Normalized Topological Distance (Euclidean)", fontsize=12)
plt.ylabel("Normalized Reconstruction Error ($1 - R^2$)", fontsize=12)
plt.colorbar(label="Composite Uniqueness Score (CUS)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()